In [ ]:
# For API calls and response parsin
import pandas as pd
import requests
from pprint import pprint

# For connecting to Supabase
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
import psycopg

## Testing the API Connection

In [ ]:
url = "https://clinicaltrials.gov/api/v2/studies"
params={"query.cond": "non-small cell lung cancer",
        "pageSize": 1000,
        "filter.advanced": "AREA[StartDate]2022"}
response = requests.get(url, params)
study = response.json()

## Inspecting the Return Object

In [ ]:
study['nextPageToken']

In [ ]:
for i in study["studies"]:
    print(i['protocolSection'])

In [ ]:
pprint(study["studies"][2]['protocolSection'])

## Identifying Key Variables in the Response Object

#### Study Identifiers

- NCT ID: study['studies'][2]['protocolSection']['identificationModule']['nctId']
- Sponsor's internal trial ID: study["studies"][2]['protocolSection']['identificationModule']['orgStudyIdInfo']['id']
- Study Acronym: study['studies'][2]['protocolSection']['identificationModule']['acronym']
- Study Title: study['studies'][2]['protocolSection']['identificationModule']['briefTitle']
- Long Title: study['studies'][2]['protocolSection']['identificationModule']['officialTitle']

#### Sponsor Information

- Lead sponsor: study['studies'][2]['protocolSection']['sponsorCollaboratorsModule']['leadSponsor']
- Collaborators: study['studies'][2]['protocolSection']['sponsorCollaboratorsModule']['collaborators']

#### Study Design

- Arms: study['studies'][2]['protocolSection']['armsInterventionsModule']['armGroups']
- Drugs: study['studies'][2]['protocolSection']['armsInterventionsModule']['interventions']
- Conditions: study['studies'][2]['protocolSection']['conditionsModule']['conditions']
- Endpoints: study['studies'][2]['protocolSection']['outcomesModule']
- Line of Therapy: study['studies'][2]['protocolSection']['descriptionModule']['briefSummary'] --> can be extracted from this text
- Main investigational Drug; study['studies'][2]['protocolSection']['descriptionModule']['briefSummary'] --> can be extracted from this text

#### Study Timeline

- Study status: study['studies'][2]['protocolSection']['statusModule']['overallStatus']
- Start date: study['studies'][2]['protocolSection']['statusModule']

In [ ]:
# NCT Number
for s in study['studies'][2]['derivedSection']:
    print(s)

In [ ]:
for a in study['studies'][2]['protocolSection']['armsInterventionsModule']['armGroups']:
    print(a['type'])

# Response Parser

Workshopping the response parser class to extrac the key datafields from the JSON response object. Rough sketch:

- Process each section individually
- Include error handling / validation logic
- Package results in a pandas dataframe

In [ ]:
# copying the response object to change variable names
api_response_json = study.copy()

# Flattener function
def flatten_study(study: dict) -> dict:
    '''Flattens the study dictionary of the JSON API response into a single dictionary to turn into a DataFrame'''
    id_mod = study['protocolSection'].get("identificationModule", {})
    sponsor_mod = study['protocolSection'].get("sponsorCollaboratorsModule", {})
    arm_mod = study['protocolSection'].get("armsInterventionsModule", {})
    cond_mod = study['protocolSection'].get("conditionsModule", {})
    outcome_mod = study['protocolSection'].get("outcomesModule", {})
    desc_mod = study['protocolSection'].get("descriptionModule", {})
    status_mod = study['protocolSection'].get("statusModule", {})

    extracted_dict = {
        "nct" : id_mod.get("nctId"),
        "study_acronym": id_mod.get("acronym"),
        "study_title": id_mod.get("briefTitle"),
        "study_long_title": id_mod.get("officialTitle"),
        "lead_sponsor": sponsor_mod.get("leadSponsor", {}).get("name"),
        "collaborators": sponsor_mod.get("collaborators"),
        "arms": arm_mod.get("armGroups"),
        "conditions": cond_mod.get("conditions"),
        "endpoints": outcome_mod.get("outcomesModule"),
        "status": status_mod.get("overallStatus"),
        "start_date": status_mod.get("statusModule")
    }
    
    return extracted_dict

# Prelim testing
results = pd.DataFrame.from_dict([flatten_study(s) for s in api_response_json['studies']])

In [ ]:
results

# Connecting to Supabase

In [ ]:
load_dotenv()

In [ ]:
client = create_engine(os.getenv("SUPABASE_PROJECT_URL"))

In [ ]:
results.to_sql(name="studies",
               con=client,
               if_exists="replace",
               index=False
               )